# LeetCode 2287 — Rearrange Characters to Make Target String bằng Python

Link đề: https://leetcode.com/problems/rearrange-characters-to-make-target-string/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **"có thể dựng bao nhiêu bản sao của `target` từ các chữ cái của `s`"** — một dạng bài *tài nguyên / công thức* quen thuộc (có bao nhiêu chữ cái thì làm được bao nhiêu bộ).
2. Dùng `collections.Counter` để đếm tần suất ký tự và duyệt `.items()`.
3. Nhận ra đáp án là **min của thương nguyên** `count_s[c] // count_target[c]` trên mọi ký tự `c` của `target` (nguyên tắc "mắt xích yếu nhất").
4. Viết 5 cách: mô phỏng lấy bớt từng bản sao, `Counter` + `min`, mảng 26 phần tử, tìm kiếm nhị phân theo đáp án `k`, và phép chia dùng `Counter` có sẵn toán tử.
5. Tránh các lỗi: dùng `count_s[c] // 1` thay vì chia theo số lần cần, quên chữ cái vắng mặt trong `s`, đếm `set(target)` làm mất số lần lặp.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `target`. Bạn có thể **lấy các ký tự từ `s`** và **sắp xếp lại** chúng để tạo các chuỗi mới. Mỗi ký tự của `s` chỉ dùng **tối đa một lần**. Trả về **số bản sao tối đa** của `target` có thể tạo được.

| Ví dụ | s | target | Output | Giải thích |
|---|---|---|---|---|
| 1 | `"ilovecodingonleetcode"` | `"code"` | `2` | có `c`×2? `c` xuất hiện 2, `o` 4, `d` 2, `e` 4 → làm được 2 bản `"code"` |
| 2 | `"abcba"` | `"abc"` | `1` | `a`×2, `b`×2, `c`×1: `c` chỉ có 1 → 1 bản |
| 3 | `"abbaccaddaeea"` | `"aaaaa"` | `1` | `a` xuất hiện 6 lần, mỗi bản cần 5 → `6 // 5 = 1` |

Constraints: `1 <= len(s) <= 100`, `1 <= len(target) <= 10`; `s` và `target` gồm chữ cái tiếng Anh thường.

In [1]:
from collections import Counter

examples = [("ilovecodingonleetcode", "code", 2), ("abcba", "abc", 1), ("abbaccaddaeea", "aaaaa", 1)]
for s, target, expected in examples:
    cs, ct = Counter(s), Counter(target)
    print(f"s={s!r}")
    print(f"  cần {dict(ct)} ; có {{{', '.join(f'{c!r}: {cs[c]}' for c in ct)}}} ; mong đợi {expected}")

s='ilovecodingonleetcode'
  cần {'c': 1, 'o': 1, 'd': 1, 'e': 1} ; có {'c': 2, 'o': 4, 'd': 2, 'e': 4} ; mong đợi 2
s='abcba'
  cần {'a': 1, 'b': 1, 'c': 1} ; có {'a': 2, 'b': 2, 'c': 1} ; mong đợi 1
s='abbaccaddaeea'
  cần {'a': 5} ; có {'a': 5} ; mong đợi 1


## 3. Kiến thức nền

### 3.1. Bảng tần suất ký tự với `collections.Counter`

`Counter("abcba")` là một từ điển `{ký tự: số lần}`. Truy cập khóa không có trả về `0` (không lỗi). `.items()` cho các cặp `(ký tự, số lần)`.

In [2]:
from collections import Counter
c = Counter("abcba")
print(c)
print(c["a"], c["z"])
print(list(c.items()))

Counter({'a': 2, 'b': 2, 'c': 1})
2 0
[('a', 2), ('b', 2), ('c', 1)]


### 3.2. Chia nguyên `//` và "mắt xích yếu nhất"

Nếu mỗi bản sao cần `need` ký tự `c` và ta có `have` ký tự `c`, ta dựng được tối đa `have // need` bản **xét riêng ký tự `c`**. Một bản sao hoàn chỉnh cần **tất cả** ký tự, nên số bản sao thực tế là **giá trị nhỏ nhất** trên mọi ký tự.

In [3]:
print(6 // 5, 4 // 2, 1 // 1, 0 // 3)
print(min(2 // 1, 4 // 1, 2 // 1, 4 // 1))

1 2 1 0
2


### 3.3. `min` với biểu thức sinh

`min(f(c) for c in iterable)` tính `f` cho mỗi phần tử rồi chọn giá trị nhỏ nhất, không tạo danh sách trung gian.

In [4]:
need = {"c": 1, "o": 1, "d": 1, "e": 1}
have = {"c": 2, "o": 4, "d": 2, "e": 4}
print(min(have[ch] // need[ch] for ch in need))

2


### 3.4. Đếm bằng mảng 26 phần tử

Với chữ cái thường, `ord(ch) - ord("a")` cho chỉ số `0..25`. Mảng `[0] * 26` thay `Counter` khi muốn dùng bộ nhớ cố định.

In [5]:
print(ord("a"), ord("c"), ord("c") - ord("a"), chr(ord("a") + 3))
cnt = [0] * 26
for ch in "abcba":
    cnt[ord(ch) - ord("a")] += 1
print(cnt[:5])

97 99 2 d
[2, 2, 1, 0, 0]


### 3.5. Phép toán trên `Counter`

`Counter` hỗ trợ phép trừ giữ phần dương: `a - b` loại bỏ `b` khỏi `a` (bỏ các số đếm `<= 0`). Điều này tiện để mô phỏng "lấy `target` ra khỏi kho" và kiểm tra xem còn đủ không.

Kiểm tra "đủ": `all(cs[c] >= ct[c] for c in ct)`.

In [6]:
a = Counter("aabbc")
b = Counter("abz")
print(a - b)                         # z bị loại vì a không có
print(all(a[ch] >= b[ch] for ch in b))
print(all(a[ch] >= 1 for ch in "abc"))

Counter({'a': 1, 'b': 1, 'c': 1})
False
True


### 3.6. Vòng lặp `while` mô phỏng

`while điều_kiện:` lặp đến khi điều kiện sai. Bài mô phỏng sẽ lặp "mỗi lần lấy đi một bản sao `target`" đến khi không đủ chữ cái.

In [7]:
n = 3
while n > 0:
    print("còn", n)
    n -= 1

còn 3
còn 2
còn 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giống như làm bánh theo công thức: một chiếc bánh `"code"` cần 1 `c`, 1 `o`, 1 `d`, 1 `e`. Kho có `c`:2, `o`:4, `d`:2, `e`:4.

- Theo `c`: làm được `2` bánh.
- Theo `o`: `4` bánh.
- Theo `d`: `2` bánh.
- Theo `e`: `4` bánh.

Làm được **ít nhất** trong các số đó: `2`. Nguyên liệu khan hiếm nhất quyết định. Với `target = "aaaaa"` (cần 5 chữ `a` mỗi bản) và kho `a`:6: `6 // 5 = 1`.

Vậy: đáp án = `min over c in target of (count_s[c] // count_target[c])`.

In [8]:
s, target = "abbaccaddaeea", "aaaaa"
cs, ct = Counter(s), Counter(target)
for ch in ct:
    print(f"{ch!r}: kho={cs[ch]}, mỗi bản cần={ct[ch]} -> làm được {cs[ch] // ct[ch]} bản")
print("đáp án:", min(cs[ch] // ct[ch] for ch in ct))

'a': kho=5, mỗi bản cần=5 -> làm được 1 bản
đáp án: 1


## 5. Cách 1 — Brute Force / baseline: mô phỏng lấy bớt từng bản sao

Lặp: nếu kho đủ các chữ cái của `target`, **trừ** đi đúng số lượng đó và tăng `copies`. Dừng khi không đủ. Rất trực quan, đúng với đề ("mỗi ký tự dùng tối đa một lần").

In [9]:
def rearrange_simulate(s, target):
    stock = Counter(s)
    need = Counter(target)
    copies = 0
    while all(stock[ch] >= need[ch] for ch in need):
        for ch in need:
            stock[ch] -= need[ch]
        copies += 1
    return copies

print(rearrange_simulate("ilovecodingonleetcode", "code"), rearrange_simulate("abcba", "abc"), rearrange_simulate("abbaccaddaeea", "aaaaa"))

2 1 1


In [10]:
from collections import Counter
from typing import List

class SolutionSimulate:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        stock = Counter(s)
        need = Counter(target)
        copies = 0
        while all(stock[ch] >= need[ch] for ch in need):
            for ch in need:
                stock[ch] -= need[ch]
            copies += 1
        return copies

print(SolutionSimulate().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(SolutionSimulate().rearrangeCharacters("abcba", "abc"))
print(SolutionSimulate().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


**Độ phức tạp:** `n = len(s)`, `m = len(target)`. Số vòng lặp tối đa `n // m`; mỗi vòng `O(k)` với `k ≤ 26` ký tự khác nhau của `target`. Tổng **Time `O(n + (n/m)·k)`**, thực chất `O(n)`; **Space `O(1)`** (tối đa 26 khóa).

**Nút thắt:** ta không cần lặp từng bản sao: chia nguyên cho đáp án trực tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

- Mỗi ký tự `c` của `target` giới hạn số bản sao tối đa là `count_s[c] // count_target[c]` (độc lập với các ký tự khác).
- Đáp án = `min` của các giới hạn đó (một bản sao cần **đủ cả bộ**).
- Nếu có ký tự của `target` vắng mặt trong `s` → `count_s[c] = 0` → giới hạn `0` → đáp án `0`. `Counter` trả `0` cho khóa vắng nên không cần xử lý riêng.
- Chia theo `count_target[c]` (không phải `1`), vì `target` có thể chứa ký tự lặp (`"aaaaa"`).

Quy tắc: **"nếu thiếu một nguyên liệu nào đó thì cả bộ không làm được; còn dư thì bao nhiêu cũng không giúp thêm."**

## 7. Các cách tiếp theo

### Cách 2 — `Counter` + `min` các thương nguyên (khuyến nghị)

In [11]:
from collections import Counter
from typing import List

class SolutionCounter:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        cs, ct = Counter(s), Counter(target)
        return min(cs[ch] // need for ch, need in ct.items())

print(SolutionCounter().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(SolutionCounter().rearrangeCharacters("abcba", "abc"))
print(SolutionCounter().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


**Độ phức tạp:** Time `O(n + m)` (dựng hai `Counter` rồi duyệt `≤ 26` khóa); Space `O(1)` (tối đa 26 khóa mỗi bảng).

### Cách 3 — Hai mảng đếm 26 phần tử

Giống Cách 2 nhưng bằng mảng; không dùng thư viện. Chỉ xét chữ cái có `need[i] > 0`.

In [12]:
from typing import List

class SolutionArray:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        have = [0] * 26
        need = [0] * 26
        for ch in s:
            have[ord(ch) - 97] += 1
        for ch in target:
            need[ord(ch) - 97] += 1
        best = len(s)                      # cận trên an toàn
        for i in range(26):
            if need[i] > 0:
                best = min(best, have[i] // need[i])
        return best

print(SolutionArray().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(SolutionArray().rearrangeCharacters("abcba", "abc"))
print(SolutionArray().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


**Độ phức tạp:** Time `O(n + m + 26)`, Space `O(26) = O(1)`.

### Cách 4 — Tìm kiếm nhị phân theo đáp án `k`

Với `k` bản sao, cần `k * need[c]` ký tự `c` cho mọi `c`. Hàm `can(k)` (kiểm tra làm được `k` bản) **đơn điệu**: nếu làm được `k` thì làm được `k - 1`. Ta tìm `k` lớn nhất có `can(k) = True` trong `[0, len(s) // len(target)]`. Dù chậm hơn Cách 2 trong bài này, đây là kỹ thuật *binary search on answer* rất hay gặp với các bài "tối đa bao nhiêu".

In [13]:
from collections import Counter
from typing import List

class SolutionBinary:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        cs, ct = Counter(s), Counter(target)

        def can(k):
            return all(cs[ch] >= k * need for ch, need in ct.items())

        lo, hi = 0, len(s) // len(target)     # k nằm trong [lo, hi]
        while lo < hi:
            mid = (lo + hi + 1) // 2           # lấy mid nghiêng lên để không lặp vô hạn
            if can(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print(SolutionBinary().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(SolutionBinary().rearrangeCharacters("abcba", "abc"))
print(SolutionBinary().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


**Độ phức tạp:** `hi ≤ n / m`; mỗi lần `can` tốn `O(k)` với `k ≤ 26`. Tổng Time `O(n + k·log(n/m))`, Space `O(1)`.

### Cách 5 — Dùng phép toán trên `Counter` (tập hợp đa trùng)

`Counter` hỗ trợ `-` giữ phần dương. Lặp `stock -= need` cho đến khi **kích thước giảm không đủ**: kiểm tra `sum((need - stock).values()) == 0`, tức không còn ký tự nào thiếu. Cách này gần với Cách 1 nhưng viết bằng toán tử cấp cao.

In [14]:
from collections import Counter
from typing import List

class SolutionOps:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        stock, need = Counter(s), Counter(target)
        copies = 0
        while not (need - stock):             # need - stock rỗng <=> kho đủ
            stock = stock - need              # lấy đi một bản
            copies += 1
        return copies

print(SolutionOps().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(SolutionOps().rearrangeCharacters("abcba", "abc"))
print(SolutionOps().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


**Độ phức tạp:** mỗi vòng `O(k)`, tối đa `n / m` vòng → Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 với `s = "abbaccaddaeea"`, `target = "aaaaa"`; và trace mô phỏng Cách 1 với `target = "code"`.

| ký tự | `count_s` | `count_target` | `count_s // count_target` |
|---|---|---|---|
| `a` | 6 | 5 | 1 |

`min` của dãy một phần tử là `1`.

In [15]:
def trace_min(s, target):
    cs, ct = Counter(s), Counter(target)
    print(f"s={s!r} target={target!r}")
    limits = []
    for ch, need in ct.items():
        lim = cs[ch] // need
        limits.append(lim)
        print(f"  ch={ch!r}: có {cs[ch]}, cần mỗi bản {need} -> {lim} bản")
    print("  min =", min(limits))
    return min(limits)

trace_min("abbaccaddaeea", "aaaaa")
trace_min("ilovecodingonleetcode", "code")

s='abbaccaddaeea' target='aaaaa'
  ch='a': có 5, cần mỗi bản 5 -> 1 bản
  min = 1
s='ilovecodingonleetcode' target='code'
  ch='c': có 2, cần mỗi bản 1 -> 2 bản
  ch='o': có 4, cần mỗi bản 1 -> 4 bản
  ch='d': có 2, cần mỗi bản 1 -> 2 bản
  ch='e': có 4, cần mỗi bản 1 -> 4 bản
  min = 2


2

In [16]:
def trace_simulate(s, target):
    stock, need, copies = Counter(s), Counter(target), 0
    print("kho ban đầu:", {c: stock[c] for c in need})
    while all(stock[c] >= need[c] for c in need):
        for c in need:
            stock[c] -= need[c]
        copies += 1
        print(f"  sau bản thứ {copies}: kho còn", {c: stock[c] for c in need})
    print("không đủ nữa -> dừng, copies =", copies)
    return copies

trace_simulate("ilovecodingonleetcode", "code")

kho ban đầu: {'c': 2, 'o': 4, 'd': 2, 'e': 4}
  sau bản thứ 1: kho còn {'c': 1, 'o': 3, 'd': 1, 'e': 3}
  sau bản thứ 2: kho còn {'c': 0, 'o': 2, 'd': 0, 'e': 2}
không đủ nữa -> dừng, copies = 2


2

## 9. Code nộp LeetCode

In [17]:
from collections import Counter
from typing import List

class Solution:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        cs, ct = Counter(s), Counter(target)
        return min(cs[ch] // need for ch, need in ct.items())

print(Solution().rearrangeCharacters("ilovecodingonleetcode", "code"))
print(Solution().rearrangeCharacters("abcba", "abc"))
print(Solution().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

2
1
1


## 10. Vì sao thuật toán đúng

- **Điều kiện cần và đủ cho `k` bản sao:** để tạo `k` bản `target`, với mỗi chữ cái `c` cần đúng `k * need[c]` ký tự `c`, và các chữ cái khác nhau không tranh nhau (mỗi ký tự thuộc đúng một chữ cái). Vì vậy `k` bản làm được ⇔ với mọi `c` trong `target`: `have[c] >= k * need[c]` ⇔ `k <= have[c] // need[c]` (do `k` nguyên).
- **Cực đại:** `k` lớn nhất thỏa với **mọi** `c` đồng thời là `min` trên `c` của `have[c] // need[c]`. Đó chính là công thức của Cách 2.
- **Sắp xếp lại tự do:** đề cho phép sắp xếp tuỳ ý nên chỉ **bộ đa tập hợp** chữ cái quan trọng, không phụ thuộc thứ tự trong `s`.
- **Biên:** nếu `c` vắng trong `s` thì `have[c] = 0` → `k = 0`. Cách 1 và 5 dừng ngay ở vòng đầu cho `0`.
- **Nhị phân:** `can(k)` đơn điệu giảm theo `k` (đúng với `k` nhỏ, sai với `k` lớn), nên tìm kiếm nhị phân tìm đúng ranh giới.

## 11. Phân tích Time / Space Complexity

`n = len(s) ≤ 100`, `m = len(target) ≤ 10`, bảng chữ cái `k ≤ 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng lấy bớt | `O(n + k·n/m)` ≈ `O(n)` | `O(1)` |
| 2. `Counter` + `min` | `O(n + m)` | `O(1)` |
| 3. Mảng 26 | `O(n + m)` | `O(1)` |
| 4. Nhị phân theo `k` | `O(n + k·log(n/m))` | `O(1)` |
| 5. Toán tử `Counter` | `O(n)` | `O(1)` |

Đọc `s` ít nhất một lần nên `O(n)` là tối ưu về bậc.

## 12. Bộ test

Ví dụ đề, edge case (thiếu hẳn một chữ cái, `target` dài hơn `s`, `target` một ký tự, ký tự lặp nhiều) và test ngẫu nhiên đối chiếu **tất cả cách** với mô phỏng thật sự.

In [18]:
import random

solvers = [SolutionSimulate(), SolutionCounter(), SolutionArray(), SolutionBinary(), SolutionOps(), Solution()]

fixed = [
    ("ilovecodingonleetcode", "code", 2),
    ("abcba", "abc", 1),
    ("abbaccaddaeea", "aaaaa", 1),
    ("a", "a", 1),                       # nhỏ nhất
    ("a", "b", 0),                       # thiếu hẳn
    ("abc", "abcd", 0),                  # target dài hơn s
    ("aaaaaaaaaa", "a", 10),             # một ký tự lặp
    ("aaaaaaaaaa", "aaa", 3),            # 10 // 3
    ("abababab", "ab", 4),
    ("abababab", "aab", 2),              # a: 4//2 = 2, b: 4//1 = 4
    ("zzzzyyyxx", "xyz", 2),             # x chỉ có 2
    ("abcdefghijklmnopqrstuvwxyz", "abc", 1),
    ("aabbcc", "abcabc", 1),
]
for s, target, expected in fixed:
    for sol in solvers:
        assert sol.rearrangeCharacters(s, target) == expected, (type(sol).__name__, s, target, expected)
print("test cố định: OK")

def reference(s, target):
    # mô phỏng thật bằng danh sách, không dùng Counter
    pool = list(s)
    copies = 0
    while True:
        trial = pool[:]
        ok = True
        for ch in target:
            if ch in trial:
                trial.remove(ch)
            else:
                ok = False
                break
        if not ok:
            return copies
        pool = trial
        copies += 1

random.seed(2287)
for _ in range(500):
    n = random.randint(1, 60)
    m = random.randint(1, 10)
    alphabet = random.choice(["ab", "abc", "abcdefgh", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(random.choice(alphabet) for _ in range(n))
    target = "".join(random.choice(alphabet) for _ in range(m))
    expected = reference(s, target)
    for sol in solvers:
        assert sol.rearrangeCharacters(s, target) == expected, (type(sol).__name__, s, target)
print("500 test ngẫu nhiên: OK")

test cố định: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chia cho `1` thay vì số lần cần (làm mất ký tự lặp)

Nếu chỉ lấy `min(cs[ch] for ch in set(target))` ta bỏ qua hoàn toàn việc mỗi bản cần nhiều ký tự giống nhau (`set` xoá số lần lặp).

In [19]:
def wrong1(s, target):
    cs = Counter(s)
    return min(cs[ch] for ch in set(target))

print("sai :", wrong1("abbaccaddaeea", "aaaaa"), "(đáng lẽ 1)")
print("đúng:", Solution().rearrangeCharacters("abbaccaddaeea", "aaaaa"))

sai : 5 (đáng lẽ 1)
đúng: 1


### Lỗi 2: Chia cho `1` thay vì `ct[ch]`

Với `target = "aab"` mỗi bản cần **2** chữ `a`; chia cho `1` coi như cần 1 chữ.

In [20]:
def wrong2(s, target):
    cs = Counter(s)
    ct = Counter(target)
    return min(cs[ch] // 1 for ch in ct)

print("sai :", wrong2("aaaabbbb", "aab"), "(min(4, 4) = 4; đáng lẽ 2)")
print("đúng:", Solution().rearrangeCharacters("aaaabbbb", "aab"))

sai : 4 (min(4, 4) = 4; đáng lẽ 2)
đúng: 2


### Lỗi 3: Lấy `len(s) // len(target)` mà không kiểm tra ký tự

Độ dài đủ chưa có nghĩa chữ cái đủ.

In [21]:
def wrong3(s, target):
    return len(s) // len(target)

print("sai :", wrong3("aaaaaaaa", "ab"), "(đáng lẽ 0 vì không có b)")
print("đúng:", Solution().rearrangeCharacters("aaaaaaaa", "ab"))

sai : 4 (đáng lẽ 0 vì không có b)
đúng: 0


### Lỗi 4: Dùng `max` thay vì `min`

Ta cần mắt xích **yếu nhất**, không phải mạnh nhất.

In [22]:
def wrong4(s, target):
    cs, ct = Counter(s), Counter(target)
    return max(cs[ch] // need for ch, need in ct.items())

print("sai :", wrong4("abcba", "abc"), "(max cho 2; đáng lẽ 1)")
print("đúng:", Solution().rearrangeCharacters("abcba", "abc"))

sai : 2 (max cho 2; đáng lẽ 1)
đúng: 1


### Lỗi 5: Duyệt các chữ cái của `s` thay vì của `target`

Chữ cái của `s` không nằm trong `target` có thể khiến `min` bằng `0`/lỗi chia cho `0`.

In [23]:
def wrong5(s, target):
    cs, ct = Counter(s), Counter(target)
    try:
        return min(cs[ch] // ct[ch] for ch in cs)     # ct[ch] có thể bằng 0
    except ZeroDivisionError as e:
        return f"ZeroDivisionError: {e}"

print("sai :", wrong5("abcba", "ab"))
print("đúng:", Solution().rearrangeCharacters("abcba", "ab"))

sai : ZeroDivisionError: integer division or modulo by zero
đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa bao nhiêu bộ / bản sao từ kho nguyên liệu" → **`min(kho[c] // cần[c])`**.
- Cùng họ: **LeetCode 1189 Maximum Number of Balloons** (`balloon` có `l`, `o` lặp 2 lần), 2287 là phiên bản tổng quát.
- Khi các giá trị cần *độ lớn* của đáp án đơn điệu → **tìm kiếm nhị phân theo đáp án**.
- `Counter` là công cụ đầu tiên cho mọi bài "đếm chữ cái sau khi sắp xếp lại".
- `Counter[x]` cho `0` khi vắng, nên tránh `KeyError` — chỉ cần chú ý chia cho `0` khi duyệt nhầm tập khóa.

## 15. Biến thể của bài

1. LeetCode 1189 — Maximum Number of Balloons: `target = "balloon"`.
2. Kho chữ cái là **danh sách từ** thay vì một chuỗi: gộp trước rồi áp dụng công thức.
3. Cho **nhiều `target`** khác nhau: tối đa tổng số bản sao (bài toán quy hoạch / tham lam).
4. Trả về **chuỗi còn lại** sau khi lấy hết các bản sao tối đa (`stock - need * k`).
5. Mỗi chữ cái có **chi phí** khác nhau và có giới hạn ngân sách (kết hợp tìm kiếm nhị phân).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | trừ từng bản sao | `O(n)` | `O(1)` | dễ hiểu nhất, học khái niệm |
| 2. `Counter` + `min` | thương nguyên nhỏ nhất | `O(n+m)` | `O(1)` | **khuyến nghị khi nộp** |
| 3. Mảng 26 | không dùng thư viện | `O(n+m)` | `O(1)` | phỏng vấn, cần kiểm soát bộ nhớ |
| 4. Nhị phân theo `k` | kiểm tra "được `k` bản?" | `O(n + k·log)` | `O(1)` | bài tối đa khi công thức không rõ |
| 5. Toán tử `Counter` | `need - stock` rỗng? | `O(n)` | `O(1)` | viết gọn bằng đại số đa tập hợp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print(Counter("hello")["l"], Counter("hello")["z"], 7 // 2, min(3 // 1, 8 // 2))` in ra gì?

**Bài 2 (tính tay).** `s = "balloonballoon"`, `target = "balloon"`. Có thể tạo bao nhiêu bản?

**Bài 3 (điền TODO).** Hoàn thành hàm trả về **ký tự gây nghẽn** (ký tự có thương nhỏ nhất).

In [24]:
def bottleneck(s, target):
    # TODO: tính thương cs[ch] // ct[ch] cho từng ch trong target, trả ch có thương nhỏ nhất
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `leftover(s, target)` trả về `Counter` các ký tự còn lại sau khi lấy hết số bản `target` tối đa.

In [25]:
def leftover(s, target):
    # TODO: tính k = số bản tối đa rồi trừ k * need
    pass

**Bài 5 (mini challenge).** Viết `max_copies_two(s, t1, t2)` trả về số bản **tối đa của `t1`** sao cho vẫn còn đủ chữ cái để làm **ít nhất một bản `t2`** (nếu không thể làm `t2` kể cả khi không làm `t1`, trả `-1`).

In [26]:
def max_copies_two(s, t1, t2):
    # TODO: thử k từ lớn xuống nhỏ, hoặc nhị phân theo k
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `hello` có 2 chữ `l`; `z` vắng → `0`; `7//2=3`; `min(3, 4) = 3`. Bài 2: `balloon` cần `b1 a1 l2 o2 n1`; chuỗi có `b2 a2 l4 o4 n2` → `2`.
- **Mức 2 (vừa):** Bài 3: dùng `min(ct, key=lambda ch: cs[ch] // ct[ch])`. Bài 4: `k = min(...)` rồi `cs - Counter({ch: k * need})` (hoặc trừ trong vòng lặp).
- **Mức 3 (sâu):** Bài 5: với mỗi `k` từ `len(s) // len(t1)` xuống `0`, trừ `k` bản `t1` rồi kiểm tra còn đủ `t2`; hoặc nhị phân vì tính đơn điệu.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
from collections import Counter

# Bài 1 và 2
assert (Counter("hello")["l"], Counter("hello")["z"], 7 // 2, min(3 // 1, 8 // 2)) == (2, 0, 3, 3)
assert Solution().rearrangeCharacters("balloonballoon", "balloon") == 2

# Bài 3
def bottleneck(s, target):
    cs, ct = Counter(s), Counter(target)
    return min(ct, key=lambda ch: cs[ch] // ct[ch])

# Bài 4
def leftover(s, target):
    k = Solution().rearrangeCharacters(s, target)
    result = Counter(s)
    for ch, need in Counter(target).items():
        result[ch] -= k * need
    return +result              # dấu + bỏ các số đếm bằng 0

# Bài 5
def max_copies_two(s, t1, t2):
    def enough(stock, t):
        need = Counter(t)
        return all(stock[c] >= need[c] for c in need)
    cs, c1 = Counter(s), Counter(t1)
    for k in range(len(s) // len(t1), -1, -1):
        rest = cs.copy()
        for ch, need in c1.items():
            rest[ch] -= k * need
        if all(v >= 0 for v in rest.values()) and enough(rest, t2):
            return k
    return -1

assert bottleneck("abcba", "abc") == "c"
assert bottleneck("aaaab", "aab") == "b"
assert leftover("abcba", "abc") == Counter({"a": 1, "b": 1})
assert leftover("ilovecodingonleetcode", "code")["o"] == 2
assert leftover("aaa", "b") == Counter({"a": 3})
assert max_copies_two("aabb", "a", "b") == 2
assert max_copies_two("aabb", "ab", "ab") == 1
assert max_copies_two("a", "a", "b") == -1
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tối đa số bộ từ kho nguyên liệu)

```python
from collections import Counter

class Solution:
    def rearrangeCharacters(self, s: str, target: str) -> int:
        cs, ct = Counter(s), Counter(target)
        return min(cs[ch] // need for ch, need in ct.items())
```

**Checklist**

- [ ] Duyệt các khóa của **`target`** (không phải của `s`)?
- [ ] Chia cho số lần cần `ct[ch]`, không chia cho `1`?
- [ ] Dùng `min`, không dùng `max`?
- [ ] Chữ cái vắng trong `s` cho `0` (nhờ `Counter`)?
- [ ] Đã thử `target` dài hơn `s`, ký tự lặp (`"aaaaa"`), và thiếu hẳn một chữ cái?
- [ ] Nếu viết nhị phân: `mid = (lo + hi + 1) // 2` để tránh lặp vô hạn?
- [ ] Signature đúng `rearrangeCharacters(self, s: str, target: str) -> int` và có `from typing import List`?